In [ ]:
#Instalasi Earth Engine API
!pip install earthengine-api folium

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='arsen-data')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
!pip install earthengine-api folium
import ee
import folium

ee.Authenticate()
ee.Initialize(project='arsen-data')

def add_ee_layer(self, ee_image_object, vis_params, name):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

#Daftar Provinsi Per Pulau
# Sumatera: ['Aceh', 'Sumatera Utara', 'Sumatera Barat', 'Riau', 'Jambi', 'Sumatera Selatan', 'Bengkulu', 'Lampung', 'Bangka Belitung', 'Kepulauan Riau']
# Kalimantan: ['Kalimantan Barat', 'Kalimantan Tengah', 'Kalimantan Selatan', 'Kalimantan Timur', 'Kalimantan Utara']
# Sulawesi: ['Sulawesi Utara', 'Gorontalo', 'Sulawesi Tengah', 'Sulawesi Barat', 'Sulawesi Selatan', 'Sulawesi Tenggara']
# Bali & Nusa Tenggara: ['Bali', 'Nusa Tenggara Barat', 'Nusa Tenggara Timur']
# Maluku: ['Maluku', 'Maluku Utara']
# Papua: ['Papua', 'Papua Barat']

provinsi_target = ['Papua', 'Papua Barat']
wilayahKajian = ee.FeatureCollection('projects/arsen-data/assets/gadm41_IDN_2') \
    .filter(ee.Filter.inList('NAME_1', provinsi_target))

dataset_tutupan = ee.ImageCollection("ESA/WorldCover/v200").first()
tutupanClip = dataset_tutupan.clip(wilayahKajian)

center = wilayahKajian.geometry().bounds().centroid(1).coordinates().getInfo()
my_map = folium.Map(location=[center[1], center[0]], zoom_start=6)

# Layer Tutupan Lahan
vis_tutupan = {'bands': ['Map']}
my_map.add_ee_layer(tutupanClip, vis_tutupan, 'Tutupan Lahan (10m)')

# Layer Batas Wilayah
empty_image = ee.Image().byte()
outlines = empty_image.paint(featureCollection=wilayahKajian, color=1, width=1)
my_map.add_ee_layer(outlines, {'palette': 'cyan'}, 'Garis Batas GADM')

my_map.add_child(folium.LayerControl())
display(my_map)

task_landcover = ee.batch.Export.image.toDrive(
  image=tutupanClip,
  description='LandCover_Pulau_Papua',
  scale=10,
  region=wilayahKajian.geometry().bounds(),
  maxPixels=1e13
)
task_landcover.start()

print("Proses Berhasil!")